# Feature Importance Workflow (Steps 3a–3c)

Run this **after** cohorts exist (Step 2; see `1_cohort_workflow.ipynb`). One cell per cohort × age_band for Step 3a and Step 3b (full grid from `py_helpers.constants.REQUIRED_COHORTS`); run Configuration and Sync once, then the cell(s) for the combination you need.

## Order of operations

1. **Configuration** — Project root, data root, cohort/age-band list (run once).
2. **Sync inputs** — Sync `gold/cohorts` from S3 to local/NVMe (idempotent).
3. **Step 3a** — MC-CV feature importance: one runnable cell per cohort × age_band. Produces `aggregated_feature_importance.csv`; checkpoint skip per cohort/age_band.
4. **Update cohort data before Step 3b** — Sync `gold/cohorts`, `gold/medical`, `gold/pharmacy` from S3 (run once before any Step 3b cell).
5. **Step 3b** — BupaR post-target analysis → `*_bupar_post_target_analysis.csv`; filter_and_refine → `cohort_feature_importance.csv`. One runnable cell per cohort × age_band.
6. **Step 3c** — Final update: strip any remaining BupaR-identified leakage from each `cohort_feature_importance.csv` before Step 4.

## Cohorts

Both cohorts use the **full set of age bands** (from `py_helpers.constants.REQUIRED_COHORTS`).

| Cohort | Target | Age bands |
|--------|--------|-----------|
| **falls** | `fall_injury_any` | Full set |
| **ed** | `ed_event` | Full set |

## Workflow: baseline FI, target leakage, model data

1. **Step 3a** — MC-CV feature importance → `aggregated_feature_importance.csv` per cohort × age_band (baseline for Step 3b).
2. **Step 3b** — BupaR post-target analysis → `is_post_target_leakage` per feature; `filter_and_refine_features.py` → clean `cohort_feature_importance.csv`.
3. **Step 3c** — Final strip of BupaR-identified leakage; Step 4 uses only `cohort_feature_importance.csv`.

## Configuration

In [ ]:
import sys
import os
from pathlib import Path
import subprocess
import logging

from IPython.display import Image, display

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
if not (PROJECT_ROOT / "3a_feature_importance").exists():
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from py_helpers.env_utils import get_data_root
from py_helpers.workflow_sync_checkpoint import (
    sync_s3_to_local,
    check_step_checkpoint_exists,
    save_step_checkpoint,
)
from py_helpers.feature_importance_heatmap import create_aggregated_fi_heatmap

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

PYTHON_BIN = Path(sys.executable)
S3_BUCKET = os.environ.get("CPIC_S3_BUCKET", "pgxdatalake")
DATA_ROOT = get_data_root()
AWS_PROFILE = os.environ.get("AWS_PROFILE")

# Age bands: 65–85 group only (65-74, 75-84)
_age_bands = ["65-74", "75-84"]

try:
    from py_helpers.constants import REQUIRED_COHORTS
    COHORTS = {k: [b for b in v if b in _age_bands] for k, v in REQUIRED_COHORTS.items()}
except ImportError:
    COHORTS = {"falls": _age_bands, "ed": _age_bands}

# Set True to overwrite/force Step 3a feature importance (rerun even when results exist)
FORCE_FEATURE_IMPORTANCE = False

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print(f"Python: {PYTHON_BIN}")
print(f"Age bands (65–85 group): {_age_bands}")
print(f"Cohorts: {COHORTS}")
print(f"Force feature importance (overwrite): {FORCE_FEATURE_IMPORTANCE}")

# Step 3a outputs base
OUTPUTS_BASE_3A = Path(os.environ.get("CPIC_FEATURE_IMPORTANCE_OUTPUTS", str(PROJECT_ROOT / "3a_feature_importance" / "outputs")))

## Sync required inputs from S3 to NVMe (idempotent)

Sync **gold/cohorts** from S3 so Step 3a can read cohort parquet from local/NVMe. **Idempotent:** `aws s3 sync` only updates changed or missing files.

In [ ]:
# Sync gold/cohorts from S3 to local/NVMe (required for 3a feature importance)
s3_cohorts = f"s3://{S3_BUCKET}/gold/cohorts/"
local_cohorts = DATA_ROOT / "gold" / "cohorts"
ok = sync_s3_to_local(s3_cohorts, local_cohorts, profile=AWS_PROFILE)
print(f"  gold/cohorts: {'OK' if ok else 'FAILED or skipped (no AWS CLI)'}")

## pgx-repository baseline (aggregated feature importance)

Current aggregated feature importance files in **pgx-repository** (used by Step 3a second pass as baseline). Run this cell to confirm row counts and feature counts before running Step 3a.

In [ ]:
# Load and display pgx-repository baseline summary (all cohort/age_band)
import sys
sys.path.insert(0, str(PROJECT_ROOT / "3a_feature_importance"))
from load_pgx_repo_fi import get_baseline_summary_df

baseline_df = get_baseline_summary_df()
display(baseline_df)

## Step 3a: MC-CV feature importance

Monte Carlo CV feature importance (CatBoost, XGBoost, XGBoost RF). **One runnable cell per cohort × age_band** so you can run and troubleshoot a single combination. Skipped when checkpoint exists for that cohort/age_band. Set **FORCE_FEATURE_IMPORTANCE = True** in Configuration to overwrite/force rerun (passes **--force** to the script). After running the age_band cells for a cohort, run that cohort’s heatmap cell to build the aggregated feature importance heatmap (feature × age band).

### (Optional) Run Step 3a for all cohorts × age bands

One cell to run Step 3a for **both cohorts** and **all age bands**. Respects checkpoint skip and **FORCE_FEATURE_IMPORTANCE** from Configuration.

In [ ]:
# Step 3a: run for all cohorts and all age bands (loop)
# Uses same checkpoint/skip and FORCE_FEATURE_IMPORTANCE as the per–age_band cells below.
step_name_3a = "3a_feature_importance"
script_3a = PROJECT_ROOT / "3a_feature_importance" / "run_mc_feature_importance.py"
for cohort, age_bands in COHORTS.items():
    for age_band in age_bands:
        if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
            print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
        else:
            print(f"Running Step 3a for {cohort}/{age_band}...")
            cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
            if FORCE_FEATURE_IMPORTANCE:
                cmd.append("--force")
            result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
            if result.returncode == 0:
                save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
            print(f"  {cohort}/{age_band} exit code: {result.returncode}")
print("Step 3a loop done.")

### Cohort 1: FALLS — one cell per age_band

In [ ]:
# Step 3a: falls / 0-12 only
step_name_3a = "3a_feature_importance"
script_3a = PROJECT_ROOT / "3a_feature_importance" / "run_mc_feature_importance.py"
cohort, age_band = "falls", "0-12"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: falls / 13-24 only
step_name_3a = "3a_feature_importance"
script_3a = PROJECT_ROOT / "3a_feature_importance" / "run_mc_feature_importance.py"
cohort, age_band = "falls", "13-24"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: falls / 25-44 only
cohort, age_band = "falls", "25-44"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: falls / 45-54 only
cohort, age_band = "falls", "45-54"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: falls / 55-64 only
cohort, age_band = "falls", "55-64"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: falls / 65-74 only
cohort, age_band = "falls", "65-74"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: falls / 75-84 only
cohort, age_band = "falls", "75-84"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: falls / 85-114 only
cohort, age_band = "falls", "85-114"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Aggregated feature importance heatmap for FALLS (run after age_band cells above)
cohort = "falls"
heatmap_path = create_aggregated_fi_heatmap(cohort, COHORTS[cohort], OUTPUTS_BASE_3A, top_n=50)
if heatmap_path and heatmap_path.exists():
    print(f"Heatmap saved: {heatmap_path}")
    display(Image(filename=str(heatmap_path)))
else:
    print("Heatmap skipped (need at least 2 age bands with aggregated CSVs).")

### Cohort 2: ED — one cell per age_band

In [ ]:
# Step 3a: ed / 0-12 only
cohort, age_band = "ed", "0-12"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: ed / 13-24 only
cohort, age_band = "ed", "13-24"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: ed / 25-44 only
cohort, age_band = "ed", "25-44"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: ed / 45-54 only
cohort, age_band = "ed", "45-54"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: ed / 55-64 only
cohort, age_band = "ed", "55-64"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: ed / 65-74 only
cohort, age_band = "ed", "65-74"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: ed / 75-84 only
cohort, age_band = "ed", "75-84"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Step 3a: ed / 85-114 only
cohort, age_band = "ed", "85-114"
if not FORCE_FEATURE_IMPORTANCE and check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
    print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3a for {cohort}/{age_band}...")
    cmd = [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age_band", age_band]
    if FORCE_FEATURE_IMPORTANCE:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

In [ ]:
# Aggregated feature importance heatmap for ED (run after age_band cells above)
cohort = "ed"
heatmap_path = create_aggregated_fi_heatmap(cohort, COHORTS[cohort], OUTPUTS_BASE_3A, top_n=50)
if heatmap_path and heatmap_path.exists():
    print(f"Heatmap saved: {heatmap_path}")
    display(Image(filename=str(heatmap_path)))
else:
    print("Heatmap skipped (need at least 2 age bands with aggregated CSVs).")

## Final combined feature importance heatmap (per cohort, all age bands)

One heatmap **per cohort**: feature × age band for that cohort. Rows = top features (union across age bands), columns = age bands for that cohort. Run after the Step 3a age_band cells for each cohort.

In [ ]:
# Combined heatmap per cohort: feature × age band for each cohort (all age bands for that cohort)
for cohort in COHORTS:
    heatmap_path = create_aggregated_fi_heatmap(cohort, COHORTS[cohort], OUTPUTS_BASE_3A, top_n=80)
    if heatmap_path and heatmap_path.exists():
        print(f"Combined heatmap for {cohort} saved: {heatmap_path}")
        display(Image(filename=str(heatmap_path)))
    else:
        print(f"Combined heatmap for {cohort} skipped (need at least 2 age bands with aggregated CSVs).")

## Step 3b: Remove Target Leakage

BupaR post-target analysis and refined `cohort_feature_importance.csv` in `3b_feature_importance_eda/outputs/`. **One runnable cell per cohort × age_band** (same as Step 3a): run the cell for the combination you want; skipped when checkpoint exists. For interactive EDA and plots use `3b_feature_importance_eda/step3b_interactive_analysis_cohort1.ipynb` … `cohort7.ipynb`.

### Update cohort data before Step 3b

Sync **gold/cohorts**, **gold/medical**, and **gold/pharmacy** from S3 so Step 3b has the latest data when building model_events (gold cohort filtered by 3a FI + admin removed). Run this cell **once before running any Step 3b cell** to keep the pipeline seamless between 3a and 3b.

In [ ]:
# Sync cohort and gold medical/pharmacy before Step 3b (idempotent)
S3_BUCKET = os.environ.get("CPIC_S3_BUCKET", "pgxdatalake")
data_root = get_data_root()
syncs = [
    (f"s3://{S3_BUCKET}/gold/cohorts/", data_root / "gold" / "cohorts"),
    (f"s3://{S3_BUCKET}/gold/medical/", data_root / "gold" / "medical"),
    (f"s3://{S3_BUCKET}/gold/pharmacy/", data_root / "gold" / "pharmacy"),
]
for s3_prefix, local_dir in syncs:
    ok = sync_s3_to_local(s3_prefix, local_dir, profile=AWS_PROFILE)
    print(f"  {local_dir.name}: {'OK' if ok else 'FAILED or skipped'}")
print("Cohort data updated. Ready for Step 3b.")

### Step 3b: falls / 0-12

In [ ]:
# Step 3b: falls / 0-12 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "falls", "0-12"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Step 3b: falls / 13-24

In [ ]:
# Step 3b: falls / 13-24 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "falls", "13-24"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Step 3b: falls / 25-44

In [ ]:
# Step 3b: falls / 25-44 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "falls", "25-44"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Step 3b: falls / 45-54

In [ ]:
# Step 3b: falls / 45-54 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "falls", "45-54"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Step 3b: falls / 55-64

In [ ]:
# Step 3b: falls / 55-64 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "falls", "55-64"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Step 3b: falls / 65-74

In [ ]:
# Step 3b: falls / 65-74 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "falls", "65-74"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Step 3b: falls / 75-84

In [ ]:
# Step 3b: falls / 75-84 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "falls", "75-84"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Step 3b: falls / 85-114

In [ ]:
# Step 3b: falls / 85-114 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "falls", "85-114"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Step 3b: ed / 0-12

In [ ]:
# Step 3b: ed / 0-12 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "ed", "0-12"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Step 3b: ed / 13-24

In [ ]:
# Step 3b: ed / 13-24 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "ed", "13-24"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Step 3b: ed / 25-44

In [ ]:
# Step 3b: ed / 25-44 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "ed", "25-44"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Step 3b: ed / 45-54

In [ ]:
# Step 3b: ed / 45-54 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "ed", "45-54"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Step 3b: ed / 55-64

In [ ]:
# Step 3b: ed / 55-64 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "ed", "55-64"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Step 3b: ed / 65-74

In [ ]:
# Step 3b: ed / 65-74 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "ed", "65-74"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Step 3b: ed / 75-84

In [ ]:
# Step 3b: ed / 75-84 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "ed", "75-84"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Step 3b: ed / 85-114

In [ ]:
# Step 3b: ed / 85-114 only
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort, age_band = "ed", "85-114"
if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
    print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
else:
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

## Step 3c: Final update to features passed into Step 4

**Required.** Final update to the feature list passed into Step 4: strip any remaining BupaR-identified post-target leakage from each `cohort_feature_importance.csv`. Step 4 uses only these CSVs to build model data. Run after all Step 3b cohort cells.

In [ ]:
# Update final model features: remove BupaR-identified target leakage from cohort_feature_importance
import pandas as pd
from py_helpers.feature_importance_eda_utils import resolve_cohort_fi_path

OUTPUTS_3B = PROJECT_ROOT / "3b_feature_importance_eda" / "outputs"
updated_count = 0
for cohort in COHORTS:
    for age_band in COHORTS[cohort]:
        age_fname = age_band.replace("-", "_")
        # Resolve cohort_feature_importance from 3b outputs, DATA_ROOT/gold, or S3 (same as Step 4/6)
        refined_path = resolve_cohort_fi_path(cohort, age_band, PROJECT_ROOT)
        bupar_path = OUTPUTS_3B / cohort / age_fname / f"{cohort}_{age_fname}_bupar_post_target_analysis.csv"
        if refined_path is None or not refined_path.exists():
            print(f"  Skip {cohort}/{age_band}: no cohort_feature_importance.csv (checked 3b/outputs, DATA_ROOT/gold/feature_importance, S3)")
            continue
        if not bupar_path.exists():
            print(f"  Skip {cohort}/{age_band}: no BupaR post-target analysis CSV")
            continue
        refined = pd.read_csv(refined_path)
        bupar = pd.read_csv(bupar_path)
        if "feature" not in refined.columns:
            print(f"  Skip {cohort}/{age_band}: no 'feature' column in refined CSV")
            continue
        leakage = set()
        if "is_post_target_leakage" in bupar.columns and "feature" in bupar.columns:
            leakage = set(bupar.loc[bupar["is_post_target_leakage"] == 1, "feature"].dropna().astype(str).tolist())
        if not leakage:
            msg = f"  {cohort}/{age_band}: no leakage features to remove"
            if cohort == "non_opioid_ed":
                msg += " (expected for polypharmacy: events only up to windowed HCG/ED)"
            print(msg)
            continue
        n_before = len(refined)
        refined_clean = refined[~refined["feature"].astype(str).isin(leakage)].copy()
        n_after = len(refined_clean)
        removed = n_before - n_after
        if removed > 0:
            refined_clean.to_csv(refined_path, index=False)
            print(f"  {cohort}/{age_band}: removed {removed} leakage features; {n_after} features saved to {refined_path.name}")
            updated_count += 1
        else:
            print(f"  {cohort}/{age_band}: leakage set had no overlap with refined features (already clean)")
if updated_count > 0:
    print(f"\nUpdated {updated_count} cohort_feature_importance file(s). Ready for Step 4.")
else:
    print("\nNo files updated (already clean or missing BupaR/refined outputs).")